In [15]:
import pandas as pd
from google.colab import drive

# 1. 掛載 Google 雲端硬碟 (執行後會跳出授權視窗，請同意授權)
drive.mount('/content/drive')

# 2. 定義您的資料夾路徑 (假設「金融資料探勘」在您的雲端硬碟根目錄)
folder_path = "/content/drive/MyDrive/金融資料探勘/"

# 3. 讀取「金融資料探勘」資料夾內的檔案
file_path = folder_path + "2022DataExport.csv"

# 因為檔名是 .csv，請使用 read_csv。若遇到中文亂碼錯誤，可將括號內改為 (file_path, encoding='big5')
df = pd.read_csv(file_path)
# 註：如果您的真實檔案其實是 Excel (task1.xlsx)，請將上面這行改成 df = pd.read_excel(folder_path + "task1.xlsx")

# 4. 新增 'File' 欄位，並在最後加上 '.csv' 副檔名
df['File'] = 'OptionsDaily_' + df['年月日'].astype(str).str.replace('/', '_') + '.csv'

# 5. 指定儲存路徑 (將處理好的檔案也存回「金融資料探勘」資料夾內)
output_path = folder_path + "2022_updated.csv"

# 6. 將處理好的資料直接儲存到雲端硬碟
df.to_csv(output_path, index=False, encoding='utf-8-sig')

# 7. 顯示結果與提示
print("前五筆資料預覽：")
print(df.head())
print(f"\n✅ 處理完成！檔案已直接儲存至：{output_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
前五筆資料預覽：
          年月日     收盤價(元)                         File
0  2022/12/30  14,137.69  OptionsDaily_2022_12_30.csv
1  2022/12/29  14,085.02  OptionsDaily_2022_12_29.csv
2  2022/12/28   14,173.1  OptionsDaily_2022_12_28.csv
3  2022/12/27  14,328.43  OptionsDaily_2022_12_27.csv
4  2022/12/26  14,285.13  OptionsDaily_2022_12_26.csv

✅ 處理完成！檔案已直接儲存至：/content/drive/MyDrive/金融資料探勘/2022_updated.csv


In [16]:
import pandas as pd
from google.colab import drive

# --- 步驟 1：掛載雲端硬碟 ---
drive.mount('/content/drive')

# --- 步驟 2：定義資料夾與讀取檔案 ---
folder_path = "/content/drive/MyDrive/金融資料探勘/"

# 讀取剛才處理好的 task1_updated.csv
df_task1 = pd.read_csv(folder_path + "2022_updated.csv")

# 將原本錯誤的這行刪掉或註解掉
# df_source3 = pd.read_csv(folder_path + "資料來源3.xlsx")

# 改用 pd.read_excel 來讀取 Excel 檔案
df_source3 = pd.read_excel(folder_path + "資料來源3.xlsx")

# --- 步驟 3：日期格式轉換 ---
# 將字串轉換為 pandas 的 datetime 格式，方便後續比對大小與計算天數
df_task1['年月日'] = pd.to_datetime(df_task1['年月日'])
df_source3['最後結算日'] = pd.to_datetime(df_source3['最後結算日'])

# --- 步驟 4：篩選「月選擇權」 ---
# 排除包含 'W' 的周選擇權
df_monthly_options = df_source3[~df_source3['契約月份'].str.contains('W', na=False)]
df_monthly_options = df_monthly_options.sort_values(by='最後結算日')

# --- 步驟 5：定義尋找最近月選擇權的函數 ---
def find_contract_info(trading_date):
    future_options = df_monthly_options[
        (df_monthly_options['最後結算日'] - trading_date).dt.days >= 1
    ]

    if not future_options.empty:
        closest_option = future_options.iloc[0]
        contract = closest_option['契約月份']
        expiry_date = closest_option['最後結算日']
        maturity = (expiry_date - trading_date).days
        return pd.Series([contract, expiry_date, maturity])
    else:
        return pd.Series([None, pd.NaT, None])

# --- 步驟 6：套用函數並新增欄位 ---
df_task1[['Contract', 'ContractExpiryDate', 'Maturity']] = df_task1['年月日'].apply(find_contract_info)

# --- 步驟 7：整理輸出格式 ---
df_task1['年月日'] = df_task1['年月日'].dt.strftime('%Y/%m/%d')
df_task1['ContractExpiryDate'] = df_task1['ContractExpiryDate'].dt.strftime('%Y-%m-%d')

# --- 步驟 8：儲存最終檔案到雲端硬碟 ---
# 將檔名指定為 task_maturity.csv
output_path = folder_path + "2022_maturity.csv"
df_task1.to_csv(output_path, index=False, encoding='utf-8-sig')

print("處理完成！前五筆資料預覽：")
print(df_task1.head())
print(f"\n✅ 檔案已成功儲存至您的雲端硬碟：{output_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
處理完成！前五筆資料預覽：
          年月日     收盤價(元)                         File  Contract  \
0  2022/12/30  14,137.69  OptionsDaily_2022_12_30.csv  202301.0   
1  2022/12/29  14,085.02  OptionsDaily_2022_12_29.csv  202301.0   
2  2022/12/28   14,173.1  OptionsDaily_2022_12_28.csv  202301.0   
3  2022/12/27  14,328.43  OptionsDaily_2022_12_27.csv  202301.0   
4  2022/12/26  14,285.13  OptionsDaily_2022_12_26.csv  202301.0   

  ContractExpiryDate  Maturity  
0         2023-01-30      31.0  
1         2023-01-30      32.0  
2         2023-01-30      33.0  
3         2023-01-30      34.0  
4         2023-01-30      35.0  

✅ 檔案已成功儲存至您的雲端硬碟：/content/drive/MyDrive/金融資料探勘/2022_maturity.csv


In [17]:
import pandas as pd
import numpy as np
from google.colab import files
import os

# 1. 檔案路徑設定
# 假設您將檔案直接上傳到 Colab 預設目錄
file_path = '/content/drive/MyDrive/金融資料探勘/2022_maturity.csv'

# 若檔案尚未上傳，可解除下方註解來喚出上傳按鈕
# print("請上傳 '2022_maturity.csv'：")
# uploaded = files.upload()
# file_path = list(uploaded.keys())[0]

if not os.path.exists(file_path):
    print(f"❌ 找不到檔案 {file_path}，請確認是否已上傳至 Colab。")
else:
    # 2. 讀取檔案
    df = pd.read_csv(file_path)

    # 3. 資料清理：移除 '年月日' 欄位為空值 (NaN) 的無效列
    df = df.dropna(subset=['年月日']).copy()

    # 將日期字串轉換為 datetime 格式，方便比對大小
    df['Date_dt'] = pd.to_datetime(df['年月日'])

    # 4. 定義 2022 年台灣銀行定期儲蓄存款(一般)機動利率
    def get_rf_rate(date):
        if date >= pd.to_datetime('2022-12-16'):
            return 0.01340
        elif date >= pd.to_datetime('2022-09-23'):
            return 0.01215
        elif date >= pd.to_datetime('2022-06-20'):
            return 0.01215
        elif date >= pd.to_datetime('2022-03-21'):
            return 0.01090
        else:
            return 0.00840

    # 5. 套用利率並新增 Rf 欄位
    df['Rf'] = df['Date_dt'].apply(get_rf_rate)

    # 移除暫存的日期格式欄位，保持原始欄位整潔
    df = df.drop(columns=['Date_dt'])

    # 6. 儲存處理後的檔案
    output_filename = '2022_maturity_with_Rf.csv'
    df.to_csv(output_filename, index=False, encoding='utf-8-sig')

    print("--- 處理完成！前五筆與後五筆資料預覽 ---")
    print(df.head())
    print("\n")
    print(df.tail())

    # --- 步驟 8：儲存最終檔案到雲端硬碟 ---
# 將檔名指定為 task_maturity.csv
output_path = folder_path + "2022_final.csv"
df_task1.to_csv(output_path, index=False, encoding='utf-8-sig')


print(f"\n✅ 檔案已成功儲存至您的雲端硬碟：{output_path}")

--- 處理完成！前五筆與後五筆資料預覽 ---
          年月日     收盤價(元)                         File  Contract  \
0  2022/12/30  14,137.69  OptionsDaily_2022_12_30.csv  202301.0   
1  2022/12/29  14,085.02  OptionsDaily_2022_12_29.csv  202301.0   
2  2022/12/28   14,173.1  OptionsDaily_2022_12_28.csv  202301.0   
3  2022/12/27  14,328.43  OptionsDaily_2022_12_27.csv  202301.0   
4  2022/12/26  14,285.13  OptionsDaily_2022_12_26.csv  202301.0   

  ContractExpiryDate  Maturity      Rf  
0         2023-01-30      31.0  0.0134  
1         2023-01-30      32.0  0.0134  
2         2023-01-30      33.0  0.0134  
3         2023-01-30      34.0  0.0134  
4         2023-01-30      35.0  0.0134  


            年月日     收盤價(元)                         File  Contract  \
241  2022/01/07  18,169.76  OptionsDaily_2022_01_07.csv  202201.0   
242  2022/01/06  18,367.92  OptionsDaily_2022_01_06.csv  202201.0   
243  2022/01/05  18,499.96  OptionsDaily_2022_01_05.csv  202201.0   
244  2022/01/04  18,526.35  OptionsDaily_2022_01